In [ ]:
# %pip install medicalmultitaskmodeling m3-sdk

In [ ]:
# @title Utilities, Setup, Data Download
import os
import uuid
import wandb
from pathlib import Path
import logfire
import torch
import numpy as np
import math
from PIL import Image
import matplotlib.pyplot as plt

# Disable Logfire integration if no token is set
logfire.configure(send_to_logfire="if-token-present", sampling=logfire.SamplingOptions.level_or_duration())

if "WANDB_API_KEY" not in os.environ:
    print("WANDB_API_KEY not found in environment variables. W&B logging will not be used.")
    os.environ["WANDB_MODE"] = "offline"  # Remove this to use M3's W&B integration
    # # Without W&B the folder for logging predictions can be set manually
    # if "MMM_default_log_folder" not in os.environ:
    #     Path(local_predictions_dir := "./local_predictions").mkdir(parents=True, exist_ok=True)
    #     print(f"Logging predictions to {local_predictions_dir}")
    #     os.environ["MMM_default_log_folder"] = local_predictions_dir

# Download demo data
(DATA_ROOT := Path(os.getenv("ML_DATA_CACHE", default="./data"))).mkdir(parents=True, exist_ok=True)
if not (dataset_path := DATA_ROOT / "bloodmnist_128.npz").exists():
    torch.hub.download_url_to_file(
        "https://zenodo.org/records/10519652/files/bloodmnist_128.npz?download=1", str(dataset_path.absolute())
    )
if not (dataset_path_3d := DATA_ROOT / "organmnist3d_64.npz").exists():
    torch.hub.download_url_to_file(
        "https://zenodo.org/records/10519652/files/organmnist3d_64.npz?download=1", str(dataset_path_3d.absolute())
    )

def visualize(cases):
    nrows, ncols = math.ceil(math.sqrt(len(cases))), math.ceil(math.sqrt(len(cases)))
    fig, axes = plt.subplots(nrows, ncols)
    images = [case["image"] for case in cases]
    captions = [case["meta"]["class_name"] for case in cases]
    for ax, img, cap in zip(axes.flat, images, captions):
        ax.imshow(img)
        ax.set_title(str(cap), fontsize=10)
        ax.axis("off")
    
    plt.show()

In [ ]:
# These environment variables are commonly set
# %env LOCAL_DEV_ENV=True
# %env MMM_LICENSE_ACCEPTED=i accept
# This one-liner imports MMM utilities relevant to interactive programming.
from mmm.interactive import configs as cfs, data, tasks, training, pipes, blocks, api

In [ ]:
# Most experiments need the same stuff. This prepares for multi-node multi-gpu training if torchrun is used.
env = cfs.EnvByConvention("finetuning").if_torchrun_prepare()

## Config

- By convention, we collect the parameters that should be configurable without code changes into a `HyperParameters` object
  - If your experiment wants to examine the impact of different encoders, it might make sense to include an encoder into the config object
- All objects in the library have config options implemented with pydantic

In [ ]:
from mmm.api.M3Model import UNICORN_ENCODER, M3_MODELS


class HyperParameters(cfs.ExperimentHyperParameters):
    foundation_model: data.DistributedPath | str = M3_MODELS[UNICORN_ENCODER]

    experiment_name: str = "finetuning_demo"
    resumable: bool = True  # Whether to resume from an existing checkpoint if available

    trainer: training.MTLTrainer.Config = training.MTLTrainer.Config(
        checkpoint_cache_folder=Path("trainer_checkpoints"),  # by default, config->result=true jobs will be resumed
        train_device="cuda",  # "cuda" or "cpu"
        # For experimentation, the steps per loop can be reduced.
        mtl_train_loop=cfs.TrainLoopConfig(max_steps=200),
        mtl_val_loop=cfs.ValLoopConfig(max_steps=100),
        max_epochs=3,
    )

In [ ]:
HyperParameters.update_schema(env)

VSCode provides auto-completion for all configuration options via JSON schema. If it does not exist, this command will create `./job_configs/finetuning.jsonc`.

In [ ]:
# In interactive environments the config is loaded from a file that is always located at ./job_configs/env_name.jsonc
config = HyperParameters.load_config(env)

## Anatomy of a multi-task model

- Our multi-task models consist of shared blocks (see `blocks.SharedBlock`) and tasks (see `tasks.MTLTask`)
- All blocks and tasks are PyTorch modules
- We start with 2D classification. A classification task requires
  - a `blocks.PyramidEncoder` which transforms an image Tensor[C, H, W] into feature maps list[Tensor[C, H, W]]
  - a `blocks.Squeezer` which transforms the feature maps list[Tensor[C, H, W]] into a latent representation Tensor[C, H, W]
  - a `tasks.ClassificationTask` which takes a latent representation, make predictions, and visualizes results

In [ ]:
# Download the model into the directory specified by env variable ML_DATA_CACHE, otherwise ~/.mmm/
foundation_model = api.M3Model(config.foundation_model, device_identifier=config.trainer.train_device)

# The model can be used as a dictionary of modules.
encoder: blocks.PyramidEncoder = foundation_model["encoder"]
squeezer: blocks.Squeezer = foundation_model["squeezer"]
encoder.freeze_all_parameters()  # freeze the encoder, only other modules (partial fine-tuning)

In [ ]:
# Inputs are batches of images like (batch_size, channels, height, width) which are between 0 and 1.
with torch.no_grad():
    test_input = torch.rand(2, 3, 64, 64).to(encoder.torch_device)
    feature_maps = encoder(test_input)
    latent_feature_map, latent_representation = squeezer(feature_maps)
    print("\n".join([f"{feat_map.shape}" for feat_map in feature_maps]), f"\nLatent: {latent_representation.shape}")

## Logging

For confidential data you should use an internal WANDB instance using `%env WANDB_BASE_URL=http://your-host:PORT/`
For logging to the official servers (including some of your training images by default), create an account at https://wandb.ai/ and be ready to paste your key into here.

The logging is integrated with our config system. All your user-configurable settings should be visible in the overview of the respective experiment:

In [ ]:
wandb_run = config.init_experiment(env)

If everything worked, you should be able to click a link with a randomly generated name for this experiment.
For now, this link should contain:

- your custom config in Workspace->report
- a structured overview of all your config values in Overview->Config

## Preparing data

In this guide, we will start multi-task classification trainings with the https://medmnist.com/ database.
You can install the available data from here: https://zenodo.org/records/10519652.

MMM uses [PyTorch dataloading](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) where each sample is a dictionary with fixed keys. The MedMNIST dataset has a fixed length. In consequence, we will use a `torch.utils.data.Dataset` to wrap it.

In [ ]:
class MedMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, class_names):
        self.images, self.labels, self.class_names = images, labels, class_names

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict:
        class_idx = self.labels[index].item()
        # Arbitrary info may only be given under the "meta" key
        return {"image": self.images[index], "class": class_idx, "meta": {"class_name": self.class_names[class_idx]}}

class_names = [
    "basophil",
    "eosinophil",
    "erythroblast",
    "immature granulocytes",
    "lymphocyte",
    "monocyte",
    "neutrophil",
    "platelet",
]
blooddata = np.load(dataset_path)
train_dataset = MedMNISTDataset(blooddata["train_images"], blooddata["train_labels"], class_names=class_names)
val_dataset = MedMNISTDataset(blooddata["val_images"], blooddata["val_labels"], class_names=class_names)
visualize([train_dataset[i] for i in range(16)])

Using a `TrainValCohort` you tell MMM which data to use for training and which for validation. It is also the place to define data augmentation.

In [ ]:
import torchvision.transforms.functional as F


def transform_pil_to_mmm(case: dict) -> dict:
    """
    MMM datasets are wrappers around PyTorch datasets that require a very specific format for each type of label.
    For classification, a dictionary with "image" and "class" keys is expected.
    """
    return {"image": F.to_tensor(Image.fromarray(case["image"]).convert("RGB")), "class": case["class"]}


# The data is encapsulated in an object that holds a training and a validation set.
def mmm_cohort(train_dataset, val_dataset) -> data.TrainValCohort:
    # More info such as class names was taken from https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py
    return data.TrainValCohort(
        data.TrainValCohort.Config(batch_size=(8, 8), num_workers=2),  # 8 train, 8 val
        train_ds=data.ClassificationDataset(
            train_dataset,
            src_transform=transform_pil_to_mmm,
            batch_transform=pipes.Alb(pipes.get_weak_default_augs()),  # Augmentations are applied only to training.
            class_names=class_names,
        ),
        val_ds=data.ClassificationDataset(val_dataset, src_transform=transform_pil_to_mmm, class_names=class_names),
    )


mmm_train_dataset = (my_cohort := mmm_cohort(train_dataset, val_dataset)).datasets[0]
mmm_training_case = mmm_train_dataset[0]
mmm_training_case["image"].shape, mmm_training_case["class"]

Extending to 3D requires to set a "group_id" for each slice. Slices with the same group_id belong to the same 3D volume. You can also skip this for now and start training with the first task only.

## Training your model

`training.MTLTrainer` is responsible for running the multi-task training loop. By default, it uses gradient accumulation to perform update steps consisting of all tasks that were added using `trainer.add_mtl_task(...)`. By default, it starts with a validation loop and runs each loop until exhaustion. The last step of a task might consist of a batch with smaller batchsize than the other steps.

In [ ]:
trainer: training.MTLTrainer = training.MTLTrainer(
    config.trainer,
    experiment_name=cfs.remove_wandb_special_chars(config.experiment_name),
    clear_checkpoints=not config.resumable,
).add_shared_blocks([foundation_model[k] for k in foundation_model.get_sharedblock_keys()])

Each `tasks.MTLTask` assembles its own architecture consisting of its own modules and the shared blocks. In the case of the `tasks.ClassificationTask`, these shared blocks are the shared encoder and the shared squeezer. Each `tasks.MTLTask` needs a unique name.

In [ ]:
mmm_task = tasks.ClassificationTask(
    hidden_dim=squeezer.get_hidden_dim(),
    args=tasks.ClassificationTask.Config(module_name="2dclassification"),
    cohort=my_cohort,
)

trainer.add_mtl_task(mmm_task)

In [ ]:
# After adding all tasks, the trainer is ready for training.
trainer.fit()

## Multi-task Fine-tuning

In [ ]:
from mmm.volume3d import Tomo3DProcessor

classes_3d = [
    "liver",
    "kidney-right",
    "kidney-left",
    "femur-right",
    "femur-left",
    "bladder",
    "heart",
    "lung-right",
    "lung-left",
    "spleen",
    "pancreas",
]
data3d = np.load(dataset_path_3d)
train_3d = MedMNISTDataset(data3d["train_images"], data3d["train_labels"], class_names=classes_3d)
val_3d = MedMNISTDataset(data3d["val_images"], data3d["val_labels"], class_names=classes_3d)

def transform_volume_to_slices(case: dict):
    npy_volume, label = case["image"], case["class"]
    npy_volume = npy_volume.astype(np.float32) / 255.0  # Normalize to [0, 1]
    patient_id = uuid.uuid4().hex[:8]  # Generate a random patient ID for grouping slices
    return [
        {
            # All image inputs are expected to be 3-channel
            "image": Tomo3DProcessor.repeat_channels(torch.from_numpy(npy_volume[..., i]).unsqueeze(0)).float(),
            "class": label,
            "meta": {"group_id": patient_id},
        }
        for i in range(npy_volume.shape[-1])
    ]

In [ ]:
# @title Adding the task is the same as for 2D

def mmm_cohort_3d(train_dataset, val_dataset) -> data.TrainValCohort:
    return data.TrainValCohort(
        data.TrainValCohort.Config(batch_size=(2, 2), num_workers=2),  # 4 train, 4 val
        train_ds=data.ClassificationDataset(
            train_dataset,
            src_transform=transform_volume_to_slices,
            batch_transform=pipes.ApplyToList(pipes.Alb(pipes.get_weak_default_augs(), replay_for_groups=True)),
            class_names=classes_3d,
            collate_fn=pipes.mtl_batch_collate,
        ),
        val_ds=data.ClassificationDataset(
            val_dataset,
            src_transform=transform_volume_to_slices,
            class_names=classes_3d,
            collate_fn=pipes.mtl_batch_collate,
        ),
    )


mmm_train_dataset_3d = (my_cohort := mmm_cohort_3d(train_3d, val_3d)).datasets[0]

mmm_task_3d = tasks.ClassificationTask(
    hidden_dim=squeezer.get_hidden_dim(),
    # Instruct the task to use the "grouper" transformer module from the foundation model to enable 3D context
    args=tasks.ClassificationTask.Config(
        module_name="3dclassification", grouper_key=cfs.GroupUsage(grouper_key="grouper")
    ),
    cohort=mmm_cohort_3d(train_3d, val_3d),
)

trainer.add_mtl_task(mmm_task_3d)

In [ ]:
config.trainer.max_epochs += 1
trainer.init_optimizer()  # reinitialize the optimizer because we changed the training setup by adding a new task
trainer.fit()

## Exporting the model

For different use cases we recommend different export methods:

- Native PyTorch export via `MTLTrainer.save_blocks_native`. This exports an `nn.ModuleDict` object which is expected by our inference utilities. This has the disadvantage that all dependencies have to be installed exactly as they were during the export because this uses `pickle` internally. This method is recommended when you have control over the inference environment (e.g. by using the same container as during training).
- ONNX export via `SharedBlock.export_to_onnx`. This exports a single shared block such as the encoder using the established ONNX standard. This is good for sharing with external users.

In [ ]:
# The trainer exports the modules in their current state. If you want to load a checkpoint first:
# trainer.load_checkpoint(Path("trainer_checkpoints/bigtraining42/bestbyvalidation-3"), load_optim_state=False)
# By default, there should be trainer_checkpoints/[experiment-name]/bestbyvalidation-[epoch] and latest folders.

module_dict = trainer.save_blocks_native(
    export_modules_path := data.DistributedPath.from_string("./all_blocks.pt.zip"),
    only_inference=True,  # Cohorts often should not be exported, `only_inference` strips those.
)
module_dict.keys()

In [ ]:
# Loading requires only one line of torch and is not specific to MMM:
exported_dict = api.M3Model(export_modules_path, "cuda:0")
with torch.inference_mode():
    some_feature_maps = exported_dict["encoder"](test_input)
    print(some_feature_maps[-1].shape)

The native PyTorch export requires the user to know how to assemble the blocks together. Alternatively, native export of individual tasks can be used to export a whole task's pipeline including the shared blocks. For this, the `save_task_native` of `MTLTrainer` can be used.

In [ ]:
trainer.save_task_native("2dclassification", Path("./task.pt"), only_inference=True)

In [ ]:
# For inference we need to apply the same preprocessing as for training (without augmentations):
test_image = val_dataset[0]["image"]
plt.title(f"Class: {class_names[val_dataset[0]['class']]}")
plt.imshow(test_image)
plt.show()

In [ ]:
# Loading again only requires one line and is not specific to MMM:
exported_task = torch.load("./task.pt", weights_only=False)
with torch.inference_mode():
    test_image_tensor = F.to_tensor(Image.fromarray(test_image).convert("RGB")).to(exported_task.task.torch_device)
    # The task module returns the logits, torch.argmax is used to get the class index for classification.
    task_output = exported_task.forward((test_image_tensor.unsqueeze(0), multiple_instance_learning_indices := None))
    # This differs for label types. For example, segmentation tasks have methods for transforming the network output:
    # SemSegTask.logits_to_probas -> SemSegTask.probas_to_preds
    # Useful resources for this are the task's docstring, and `training_step` and visualization methods.
mmm_task.class_names[torch.argmax(task_output)], torch.max(task_output).item()